# 01. Preprocessing

この notebook は前処理を扱います。

主題:

- logical table の読み方
- source table から household-level feature を作る責務
- campaign window、pre-treatment window、post-treatment outcome window
- transform、aggregation、encoding、standardization
- feature semantics との接続

因果推論上の背景: 前処理は単なる整形ではありません。いつ測った変数か、treatment より前か後か、analysis unit は何か、という設計が交絡調整と estimand の意味を決めます。

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

REPOSITORY_MARKER = "pyproject.toml"
cwd = Path.cwd().resolve()
for candidate in (cwd, *cwd.parents):
    if (candidate / REPOSITORY_MARKER).exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("repository root was not found")

ARTICLE_ROOT = PROJECT_ROOT
SRC_DIR = PROJECT_ROOT / "src"
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
for path in (SRC_DIR,):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

ARTICLE_ROOT

## Preprocessing の型分解

| 型・関数 | package | 役割 |
|---|---|---|
| `LogicalTableDataLoader` | `ariadne.etl.registry` / `ariadne.etl.registry` | dataset registry から logical table を読む |
| `FeatureConfig` | `ariadne.preprocessing.discovery.config` | discovery 用 feature specification |
| `CompleteJourneyPreprocessor` | `ariadne.preprocessing.discovery.builder` | discovery input を作る中心 |
| `FeatureBuilder` | `ariadne.preprocessing.inference.builder` | inference input を作る中心 |
| `FeatureSemanticsCatalog` | `ariadne.preprocessing.common` | feature の意味を stage 横断で検証する |

Why so?: discovery と inference では必要な列や window が違います。しかし、treatment/outcome/covariate の意味は横断的に整合している必要があります。

In [ ]:
from ariadne.infrastructure.config import load_yaml_mapping
from ariadne.preprocessing.discovery.config import load_feature_config as load_discovery_feature_config
from ariadne.preprocessing.inference.config import load_feature_config as load_inference_feature_config

discovery_feature_path = PROJECT_ROOT / "configs" / "preprocessing" / "discovery_features.yaml"
inference_feature_path = PROJECT_ROOT / "configs" / "preprocessing" / "inference_features.yaml"

discovery_features = load_discovery_feature_config(discovery_feature_path)
inference_features = load_inference_feature_config(inference_feature_path)

pd.DataFrame([
    {"side": "discovery", "tables": len(discovery_features.tables), "features": len(discovery_features.all_features())},
    {"side": "inference", "tables": len(inference_features.tables), "aggregations": len(inference_features.aggregations)},
])

## Logical tables

file path をコードに直書きせず、dataset YAML と feature config の logical table 名で読む構成です。これにより、入力ファイルの物理配置と分析ロジックを分離します。

In [ ]:
pd.DataFrame([
    {
        "table_alias": name,
        "dataset_entry": spec.name,
        "household_key": spec.household_key,
        "week": spec.week,
    }
    for name, spec in discovery_features.tables.items()
])

## Discovery feature specification

discovery 用 feature は graph node 候補です。各 feature には source table、source column、transform、role、background tier が含まれます。

因果探索上の意味: background tier は時間順序制約に使われます。例えば treatment 後の outcome から baseline へ backward edge を許すと、時間的に不自然な graph が出やすくなります。

In [ ]:
feature_rows = []
for spec in discovery_features.all_features()[:20]:
    feature_rows.append({
        "name": spec.name,
        "source_table": spec.source_table,
        "source_column": spec.source_column,
        "transform": spec.transform,
        "role": spec.role,
        "background_tier": spec.background_tier,
        "used_in_discovery": spec.used_in_discovery,
    })
pd.DataFrame(feature_rows)

## Standardization と collinearity pruning

多くの discovery algorithm は scale や冗長列の影響を受けます。そのため、定数列を落とし、高相関列を落としてから z-score 標準化します。下のセルは小さな合成 frame で同じ考え方を確認します。

In [ ]:
from ariadne.preprocessing.common import drop_collinear_columns

rng = np.random.default_rng(42)
x = rng.normal(size=100)
frame = pd.DataFrame({
    "x": x,
    "almost_x": x + rng.normal(scale=1e-4, size=100),
    "z": rng.normal(size=100),
    "constant": 1.0,
})
non_constant = frame.loc[:, frame.std(axis=0) > 0]
retained, dropped = drop_collinear_columns(non_constant, collinearity_threshold=0.995)
standardized = (retained - retained.mean(axis=0)) / retained.std(axis=0)

pd.DataFrame({
    "retained_columns": [list(retained.columns)],
    "dropped_columns": [dropped.to_dict("records")],
    "standardized_means": [standardized.mean().round(6).to_dict()],
})

## Preprocessing での因果推論上の注意

- `analysis_unit` を固定する。ここでは household-level analysis が基本です。
- treatment より前に測られた covariate と、treatment 後に測られた outcome を混ぜない。
- post-treatment variable を交絡調整に入れると estimand が変わる可能性がある。
- 欠損補完、カテゴリ変換、window 定義は統計処理ではなく causal design の一部として扱う。

反対仮説: 前処理は単なる機械的整形だ、という見方もあります。しかし causal inference では、いつ測ったか、何の proxy か、どの単位に集約したかが識別仮定に直結します。